In [2]:
import pandas as pd
import pickle
import time
from collections import defaultdict

from funciones import obtener_comunas, ensure_dir
from modelos import modelo_sin_limite_sparse, modelo_sin_limite_opti


# ============================================================
# CONFIG
# ============================================================

EPSILON = 0.1
K_TOTAL = 28
M = 100

#DATA_DIR = "data_chile_distrito_censal"
DATA_DIR = "DataChileMANZANAS/data_chile_2024_procesada/caso_A_principal"

PATH_COMUNAS = f"{DATA_DIR}/comunas_chile_2024_caso_A_principal.xlsx"
PATH_DISTANCIAS = f"{DATA_DIR}/distancias_chile_2024_caso_A_principal.xlsx"
PATH_S = f"{DATA_DIR}/s_nuevo_chile_2024_caso_A_principal.pkl"

OUT_DIR = "diagnostico_regiones"
ensure_dir(OUT_DIR)

OUT_RESUMEN = f"{OUT_DIR}/resumen_factibilidad_regiones.csv"


# ============================================================
# CARGAR DATOS
# ============================================================

print("\n==============================")
print("LEYENDO DATOS")
print("==============================")

comunas = pd.read_excel(PATH_COMUNAS)
#distancias = pd.read_excel(PATH_DISTANCIAS)

with open(PATH_S, "rb") as f:
    dict_s_base = pickle.load(f)

dict_s = defaultdict(lambda: [[]], dict_s_base)

print("comunas:", comunas.shape)
#print("distancias:", distancias.shape)
print("claves dict_s:", len(dict_s_base))


# ============================================================
# REGIONES Y K REGIONAL
# ============================================================

regiones = sorted(comunas["region"].unique())

pop_total = comunas["poblacion2017"].sum()

# Si quieres fijar K manual por región, edita este diccionario.
# Si una región no aparece aquí, se calcula proporcional a población.
K_POR_REGION = {
    "arica_y_parinacota": 2,
    "tarapaca": 2,
    "antofagasta": 2,
    "atacama": 2,
    "coquimbo": 2,
    "valparaiso": 2,
    "metropolitana_de_santiago": 7,
    "libertador_general_bernardo_ohiggins": 2,
    "maule": 2,
    "nuble": 2,
    "biobio": 3,
    "araucania": 2,
    "los_rios": 2,
    "los_lagos": 2,
    #"aysen_del_general_carlos_ibanez_del_campo": 2,
    #"magallanes_y_de_la_antartica_chilena": 2,
}
def obtener_K_region(region):
    if region in K_POR_REGION:
        return K_POR_REGION[region]

    pop_region = comunas.loc[
        comunas["region"] == region,
        "poblacion2017"
    ].sum()

    return max(1, round(K_TOTAL * pop_region / pop_total))


# ============================================================
# FUNCIÓN AUXILIAR: FILTRAR DATA POR REGIÓN
# ============================================================

def preparar_region(region):
    R_region = obtener_comunas(comunas, region)
    R_set = set(R_region)

    comunas_region = (
        comunas[comunas["comuna"].isin(R_region)]
        .copy()
        .reset_index(drop=True)
    )

    # distancias_region = (
    #     distancias[distancias["comuna"].isin(R_region)]
    #     [["comuna"] + R_region]
    #     .copy()
    #     .reset_index(drop=True)
    # )

    # Filtramos pares principales dentro de la región.
    # Si un camino usa un nodo intermedio fuera de la región,
    # el modelo sparse lo maneja con path_block.
    dict_s_region_base = {
        (a, b): v
        for (a, b), v in dict_s_base.items()
        if a in R_set and b in R_set
    }

    dict_s_region = defaultdict(lambda: [[]], dict_s_region_base)

    return R_region, comunas_region, dict_s_region


# ============================================================
# LOOP POR REGIÓN
# ============================================================

resultados = []

for region in regiones:
    print("\n\n======================================")
    print("REGIÓN:", region)
    print("======================================")

    try:
        R_reg, comunas_reg, dict_s_reg = preparar_region(region)

        K_reg = obtener_K_region(region)
        M_reg = min(M, len(R_reg))

        pop_reg = comunas_reg["poblacion2017"].sum()

        print("N unidades:", len(R_reg))
        print("Población:", pop_reg)
        print("K regional:", K_reg)
        print("M regional:", M_reg)

        if len(R_reg) < K_reg:
            print("[SKIP] Menos unidades que K.")
            resultados.append({
                "region": region,
                "n_unidades": len(R_reg),
                "poblacion": pop_reg,
                "K": K_reg,
                #"M": M_reg,
                "modelo": "sin_limite",
                "status": "skip_n_menor_K",
                "tiempo_seg": None
            })
            continue

        # ----------------------------------------------------
        # 1. SIN LÍMITE
        # ----------------------------------------------------
        print("\n--- Probando SIN límite ---")

        t0 = time.time()

        # modelo_sl = modelo_sin_limite_sparse(
        #     epsilon=EPSILON,
        #     R=R_reg,
        #     K=K_reg,
        #     dict_s=dict_s_reg,
        #     comunas=comunas_reg,
        #     distancias=distancias_reg,
        #     M=M_reg,
        #     usar_contiguidad=True
        # )

        modelo_sl = modelo_sin_limite_opti(
            epsilon=EPSILON,
            R=R_reg,
            K=K_reg,
            dict_s=dict_s_reg,
            comunas=comunas_reg#,
            #distancias=distancias_reg
        )

        tiempo_sl = time.time() - t0

        resultados.append({
            "region": region,
            "n_unidades": len(R_reg),
            "poblacion": pop_reg,
            "K": K_reg,
            #"M": M_reg,
            "modelo": "sin_limite",
            "status": "factible" if modelo_sl is not None else "no_factible",
            "tiempo_seg": tiempo_sl
        })


    except Exception as e:
        print("[ERROR]", region, e)

        resultados.append({
            "region": region,
            "n_unidades": None,
            "poblacion": None,
            "K": None,
            "M": None,
            #"modelo": "error",
            "status": str(e),
            "tiempo_seg": None
        })

    # Guardado parcial por seguridad
    pd.DataFrame(resultados).to_csv(OUT_RESUMEN, index=False)
    print("[OK] resumen parcial guardado")


# ============================================================
# GUARDAR RESUMEN FINAL
# ============================================================

df_res = pd.DataFrame(resultados)
df_res.to_csv(OUT_RESUMEN, index=False)

print("\n======================================")
print("FIN EXPERIMENTO")
print("======================================")
print(df_res)
print("[OK]", OUT_RESUMEN)


LEYENDO DATOS
comunas: (2520, 3)
claves dict_s: 6350400


REGIÓN: antofagasta
N unidades: 64
Población: 618389
K regional: 2
M regional: 64

--- Probando SIN límite ---
Set parameter Username
Set parameter LicenseID to value 2819129
Academic license - for non-commercial use only - expires 2027-05-06
Set parameter Method to value 2
Set parameter Crossover to value 0
Set parameter Threads to value 4
Set parameter SoftMemLimit to value 15
Set parameter OutputFlag to value 1
La cantidad de centros es 2
Población total: 618389
phat: 309194.5
Rango permitido: 278275.05 340113.95
Restricciones path: 11583
Paths omitidos por k fuera de R: 28
Variables: 4160
Restricciones: 15936
Gurobi Optimizer version 12.0.1 build v12.0.1rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1355U, instruction set [SSE2|AVX|AVX2]
Thread count: 10 physical cores, 12 logical processors, using up to 4 threads

Non-default parameters:
SoftMemLimit  15
Method  2
Crossover  0
Threads  4

O

---

In [1]:
import pandas as pd
import pickle
import json
import gurobipy as gp
from collections import defaultdict, Counter

from funciones import obtener_comunas, obtener_region, pob, extraer_prob_centros
from DataChile.chile_data import regiones


# ============================================================
# CONFIG
# ============================================================

PATH_COMUNAS = "DataChileMANZANAS/data_chile_2024_procesada/caso_A_principal/comunas_chile_2024_caso_A_principal.xlsx"
PATH_S = "DataChileMANZANAS/data_chile_2024_procesada/caso_A_principal/s_nuevo_chile_2024_caso_A_principal.pkl"

PATH_MODELO_LP = "datos_modelo/modelo_chile_censal_eps_0.63000.lp"
PATH_VALORES = "datos_modelo/valores_chile_censal_eps_0.63000.json"

K = 28
EPSILON = 0.63


# ============================================================
# CARGAR DATA
# ============================================================

print("\n==============================")
print("CARGANDO DATA")
print("==============================")

comunas = pd.read_excel(PATH_COMUNAS)

with open(PATH_S, "rb") as f:
    dict_s_base = pickle.load(f)

dict_s = defaultdict(lambda: [[]], dict_s_base)

modelo_pl = gp.read(PATH_MODELO_LP)

with open(PATH_VALORES, "r", encoding="utf-8") as f:
    valores_raw = json.load(f)

valores = {k.replace(" ", "_"): v for k, v in valores_raw.items()}

print("comunas:", comunas.shape)
print("dict_s keys:", len(dict_s_base))


# ============================================================
# CONSTRUIR R
# ============================================================

R_por_region = {}

for region in sorted(comunas["region"].unique()):
    R_por_region[region] = obtener_comunas(comunas, region)

R = sum(R_por_region.values(), [])

print("Total R:", len(R))
print("Total comunas archivo:", len(comunas))

faltan = set(comunas["comuna"]) - set(R)
print("Faltan en R:", len(faltan))

if faltan:
    print(list(faltan)[:20])


# ============================================================
# EXTRAER CENTROS FIJOS Y FRACCIONARIOS DESDE PL
# ============================================================

def centros_fijados_desde_modelo(modelo, valores):
    centros = []

    for v in modelo.getVars():
        if v.VarName.startswith("centros_j") and valores.get(v.VarName, 0) == 1.0:
            centro = v.VarName[v.VarName.find("[") + 1:v.VarName.find("]")]
            centros.append(centro)

    return centros


count_centros_fijados, centros_frac, top_centros_frac = extraer_prob_centros(
    modelo_pl,
    K,
    valores
)

centros_fijados = centros_fijados_desde_modelo(modelo_pl, valores)

print("\n==============================")
print("CENTROS DEL PL")
print("==============================")

print("Centros fijados según extraer_prob_centros:", count_centros_fijados)
print("Centros fijados extraídos:", len(centros_fijados))
print("Centros fraccionarios:", len(centros_frac))
print("Centros a samplear:", K - len(centros_fijados))

print("\nCentros fijados:")
for c in centros_fijados:
    print(c, "|", obtener_region(comunas, c), "| pob:", pob(comunas, c))




CARGANDO DATA
Set parameter Username
Set parameter LicenseID to value 2819129
Academic license - for non-commercial use only - expires 2027-05-06
Read LP format model from file datos_modelo/modelo_chile_censal_eps_0.63000.lp
Reading time = 14.44 seconds
: 5416706 rows, 644854 columns, 12742930 nonzeros
comunas: (2520, 3)
dict_s keys: 6350400
Total R: 2520
Total comunas archivo: 2520
Faltan en R: 0
Hay 24 centros fijados por el Modelo

Hay 14 comunas con peso positivo


CENTROS DEL PL
Centros fijados según extraer_prob_centros: 24
Centros fijados extraídos: 24
Centros fraccionarios: 14
Centros a samplear: 4

Centros fijados:
dist_220109 | antofagasta | pob: 19885
dist_220201 | antofagasta | pob: 15
dist_1520108 | arica_y_parinacota | pob: 6
dist_330401 | atacama | pob: 8400
dist_810303 | biobio | pob: 16762
dist_410217 | coquimbo | pob: 30695
dist_410408 | coquimbo | pob: 332
dist_912004 | la_araucania | pob: 3196
dist_611001 | libertador_general_bernardo_ohiggins | pob: 579
dist_10102

In [2]:
from itertools import combinations

def cumple_balance_regional_necesario(R, centros_total, comunas, epsilon):
    pobl = {i: pob(comunas, i) for i in R}
    phat = sum(pobl.values()) / len(centros_total)

    pop_region = defaultdict(float)
    centros_region = Counter()

    for i in R:
        pop_region[obtener_region(comunas, i)] += pobl[i]

    for c in centros_total:
        centros_region[obtener_region(comunas, c)] += 1

    for r, pop_r in pop_region.items():
        m = centros_region.get(r, 0)

        cap_min = m * phat * (1 - epsilon)
        cap_max = m * phat * (1 + epsilon)

        if not (cap_min <= pop_r <= cap_max):
            return False

    return True


print("\n==============================")
print("CENTROS FRACCIONARIOS")
print("==============================")
from itertools import combinations
from modelos import modelo_centros_fijos_con_limite

# ============================================================
# BUSQUEDA DIRIGIDA POR REGIÓN
# ============================================================

candidatos_frac = [c for c, p in centros_frac]

frac_por_region = defaultdict(list)

for c in candidatos_frac:
    frac_por_region[obtener_region(comunas, c)].append(c)

print("\n==============================")
print("FRACCIONARIOS POR REGIÓN")
print("==============================")

for r, cs in sorted(frac_por_region.items()):
    print(r, len(cs))
    for c in cs:
        print(" ", c, "| pob:", pob(comunas, c))


# Según diagnóstico regional:
# Biobío necesita +1 centro.
# RM necesita +3 centros.
regiones_requeridas = {
    "biobio": 1,
    "metropolitana_de_santiago": 3,
}

listas_combos = []

for r, q in regiones_requeridas.items():
    candidatos_r = frac_por_region.get(r, [])

    if len(candidatos_r) < q:
        raise ValueError(
            f"No hay suficientes fraccionarios en {r}: "
            f"necesito {q}, hay {len(candidatos_r)}"
        )

    listas_combos.append(list(combinations(candidatos_r, q)))


combinaciones_dirigidas = []

for combo_biobio in listas_combos[0]:
    for combo_rm in listas_combos[1]:
        comb = list(combo_biobio) + list(combo_rm)
        centros_total = centros_fijados + comb

        if cumple_balance_regional_necesario(
            R=R,
            centros_total=centros_total,
            comunas=comunas,
            epsilon=EPSILON
        ):
            combinaciones_dirigidas.append(comb)

print("\nCombinaciones dirigidas válidas:", len(combinaciones_dirigidas))

for idx, comb in enumerate(combinaciones_dirigidas[:20], start=1):
    print("\nCombo", idx)
    for c in comb:
        print(" ", c, "|", obtener_region(comunas, c), "| pob:", pob(comunas, c))


# ============================================================
# RESOLVER IP CON COMBINACIONES DIRIGIDAS
# ============================================================

modelo_ip = None
centros_total_factible = None

for idx, comb in enumerate(combinaciones_dirigidas, start=1):

    print("\n==============================")
    print(f"PROBANDO COMBO DIRIGIDO {idx}/{len(combinaciones_dirigidas)}")
    print("==============================")

    centros_total = centros_fijados + comb

    print("Centros agregados:")
    for c in comb:
        print(c, "|", obtener_region(comunas, c), "| pob:", pob(comunas, c))

    modelo_ip = modelo_centros_fijos_con_limite(
        epsilon=EPSILON,
        R=R,
        C=centros_total,
        dict_s=dict_s,
        comunas=comunas,
        verbose=True
    )

    if modelo_ip:
        print("\n[OK] IP factible")
        centros_total_factible = centros_total
        break

    print("[NO] Infactible con este combo")


if modelo_ip:
    modelo_ip.write("datos_modelo/modelo_ip_centros_manual_dirigido.lp")

    valores_ip = {
        v.VarName: v.X
        for v in modelo_ip.getVars()
    }

    with open("datos_modelo/valores_ip_centros_manual_dirigido.json", "w") as f:
        json.dump(valores_ip, f)

    with open("datos_modelo/centros_manual_dirigido.txt", "w", encoding="utf-8") as f:
        for c in centros_total_factible:
            f.write(c + "\n")

    print("[OK] Guardado modelo_ip_centros_manual_dirigido.lp")
    print("[OK] Guardado valores_ip_centros_manual_dirigido.json")
    print("[OK] Guardado centros_manual_dirigido.txt")

else:
    print("\n[ERROR] Ninguna combinación dirigida fue factible.")


CENTROS FRACCIONARIOS

FRACCIONARIOS POR REGIÓN
antofagasta 1
  dist_210303 | pob: 884
biobio 2
  dist_831305 | pob: 475
  dist_810702 | pob: 12609
metropolitana_de_santiago 11
  dist_1340206 | pob: 5478
  dist_1340409 | pob: 5003
  dist_1312701 | pob: 6738
  dist_1311207 | pob: 20828
  dist_1311301 | pob: 11999
  dist_1311410 | pob: 21383
  dist_1311921 | pob: 15340
  dist_1350104 | pob: 1990
  dist_1312802 | pob: 17042
  dist_1320303 | pob: 2917
  dist_1350404 | pob: 1671

Combinaciones dirigidas válidas: 0

[ERROR] Ninguna combinación dirigida fue factible.


In [3]:
from collections import Counter, defaultdict
from itertools import product

def resumen_regional_con_k(R, centros_fijados, centros_frac, comunas, epsilon, K=28):
    pobl = {i: pob(comunas, i) for i in R}
    phat = sum(pobl.values()) / K

    pop_region = defaultdict(float)
    for i in R:
        pop_region[obtener_region(comunas, i)] += pobl[i]

    centros_fijos_region = Counter(obtener_region(comunas, c) for c in centros_fijados)

    frac_region = defaultdict(list)
    for c, p in centros_frac:
        frac_region[obtener_region(comunas, c)].append((c, p))

    filas = []

    for r in sorted(pop_region):
        pop = pop_region[r]
        fijos = centros_fijos_region.get(r, 0)

        posibles_m = []
        for m_total in range(fijos, fijos + len(frac_region.get(r, [])) + 1):
            cap_min = m_total * phat * (1 - epsilon)
            cap_max = m_total * phat * (1 + epsilon)

            if cap_min <= pop <= cap_max:
                posibles_m.append(m_total)

        filas.append({
            "region": r,
            "poblacion": pop,
            "fijos": fijos,
            "frac_disponibles": len(frac_region.get(r, [])),
            "m_totales_posibles": posibles_m,
            "agregar_posibles": [m - fijos for m in posibles_m],
        })

    df = pd.DataFrame(filas)
    print("phat K=28:", phat)
    print(df)

    return df, frac_region

df_reg, frac_region = resumen_regional_con_k(
    R=R,
    centros_fijados=centros_fijados,
    centros_frac=centros_frac,
    comunas=comunas,
    epsilon=EPSILON,
    K=28
)

phat K=28: 632040.3214285715
                                  region  poblacion  fijos  frac_disponibles  \
0                            antofagasta   618389.0      2                 1   
1                     arica_y_parinacota   237430.0      1                 0   
2                                atacama   294897.0      1                 0   
3                                 biobio  1602659.0      1                 2   
4                               coquimbo   817236.0      2                 0   
5                           la_araucania  1003546.0      1                 0   
6   libertador_general_bernardo_ohiggins   981737.0      1                 0   
7                              los_lagos   630637.0      1                 0   
8                               los_rios   375343.0      1                 0   
9                                  maule  1114486.0      3                 0   
10             metropolitana_de_santiago  7291744.0      4                11   
11         

In [ ]:

# ============================================================
# DIAGNÓSTICO REGIONAL DE CENTROS
# ============================================================

def diagnostico_centros_por_region(R, C, comunas, epsilon):
    """
    Diagnóstico necesario para modelo con límite regional.

    Como el modelo bloquea asignaciones entre regiones, cada región debe tener
    suficientes centros para cubrir su propia población.
    """

    pobl = {i: pob(comunas, i) for i in R}
    phat = sum(pobl.values()) / len(C)

    region_i = {i: obtener_region(comunas, i) for i in R}
    region_c = {j: obtener_region(comunas, j) for j in C}

    pop_region = defaultdict(float)

    for i in R:
        pop_region[region_i[i]] += pobl[i]

    centros_region = Counter(region_c.values())

    filas = []

    for r in sorted(pop_region):
        pop_r = pop_region[r]
        m_r = centros_region.get(r, 0)

        cap_min = m_r * phat * (1 - epsilon)
        cap_max = m_r * phat * (1 + epsilon)

        filas.append({
            "region": r,
            "poblacion_region": pop_r,
            "centros_region": m_r,
            "cap_min_region": cap_min,
            "cap_max_region": cap_max,
            "ok_necesario": cap_min <= pop_r <= cap_max
        })

    df = pd.DataFrame(filas)

    print("\n==============================")
    print("DIAGNÓSTICO REGIONAL")
    print("==============================")

    print("K usado:", len(C))
    print("phat:", phat)
    print("epsilon:", epsilon)
    print("rango por distrito:", phat * (1 - epsilon), phat * (1 + epsilon))

    print(df)

    problemas = df[df["ok_necesario"] == False]

    if len(problemas) > 0:
        print("\n[PROBLEMA] Hay regiones que hacen infactible cualquier solución con estos centros:")
        print(problemas)
    else:
        print("\n[OK] A nivel de capacidad regional, estos centros podrían ser factibles.")

    return df


df_diag_fijos = diagnostico_centros_por_region(
    R=R,
    C=centros_fijados,
    comunas=comunas,
    epsilon=EPSILON
)


# ============================================================
# DIAGNÓSTICO PARA UNA MUESTRA COMPLETA
# ============================================================
# Si ya tienes una muestra de centros sampleados, pégala acá.
# Si no, esto solo diagnostica los centros fijados.

centros_sampleados = [
    # "dist_xxx",
    # "dist_yyy",
]

if centros_sampleados:
    centros_total = centros_fijados + centros_sampleados

    print("\n==============================")
    print("DIAGNÓSTICO CENTROS COMPLETOS")
    print("==============================")

    print("Total centros:", len(centros_total))

    df_diag_total = diagnostico_centros_por_region(
        R=R,
        C=centros_total,
        comunas=comunas,
        epsilon=EPSILON
    )
else:
    print("\nNo se diagnosticó muestra completa porque centros_sampleados está vacío.")

###